In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# PR11: Advanced ML & Model Optimization (Wine Quality Dataset)
file_path = 'WineQT.csv'
df = pd.read_csv(file_path)
print("Dataset loaded successfully! Shape:", df.shape)
display(df.head(3))

# Convert quality into a binary classification target (Good quality >= 6, else 0)
y = (df['quality'] >= 6).astype(int)
X = df.drop(columns=['quality', 'Id'], errors='ignore')   # 'Id' is just a row number

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 5-fold cross-validation used everywhere below
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Three models with DEFAULT parameters (SVM needs scaling, so it is wrapped in a pipeline)
models = {
    'SVM': Pipeline([('scaler', StandardScaler()), ('svm', SVC())]),
    'Random Forest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42)
}

results = []   # [model, accuracy before, F1 before]
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append([name, accuracy_score(y_test, pred), f1_score(y_test, pred)])
    print(f"{name} (default) - 5-fold CV scores: {cv_scores.round(3)} | Mean: {cv_scores.mean():.3f}")

In [ ]:
# Grid Search: parameter grids for each model
param_grids = {
    'SVM': {'svm__C': [0.1, 1, 10], 'svm__kernel': ['linear', 'rbf']},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]},
    'AdaBoost': {'n_estimators': [50, 100, 200], 'learning_rate': [0.1, 0.5, 1.0]}
}

for i, (name, model) in enumerate(models.items()):
    grid = GridSearchCV(model, param_grids[name], cv=kf, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    cv_scores = cross_val_score(best, X_train, y_train, cv=kf, scoring='accuracy')
    pred = best.predict(X_test)
    results[i] += [accuracy_score(y_test, pred), f1_score(y_test, pred)]
    print(f"{name} (tuned) - Best params: {grid.best_params_}")
    print(f"   5-fold CV scores: {cv_scores.round(3)} | Mean: {cv_scores.mean():.3f}")

In [ ]:
# Before vs After optimization table
res_df = pd.DataFrame(results, columns=['Model', 'Accuracy (Before)', 'F1 (Before)',
                                        'Accuracy (After)', 'F1 (After)']).round(4)
res_df['Accuracy Gain'] = (res_df['Accuracy (After)'] - res_df['Accuracy (Before)']).round(4)
print("--- Before vs After Optimization ---")
display(res_df)

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(res_df)); w = 0.35
for ax, metric in zip(axes, ['Accuracy', 'F1']):
    ax.bar(x - w/2, res_df[f'{metric} (Before)'], w, label='Before (default)')
    ax.bar(x + w/2, res_df[f'{metric} (After)'], w, label='After (tuned)')
    ax.set_xticks(x); ax.set_xticklabels(res_df['Model'])
    ax.set_ylim(0.5, 1); ax.set_ylabel(metric); ax.set_title(f'{metric}: Before vs After Tuning')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()